# ONNX Accuracy & Size

## Import the required modules

In [10]:
import numpy as np
import onnxruntime as ort
import os
import torch
import torchaudio

from msc_dataset import MSCDataset

## Create the Test Dataset

In [11]:
CLASSES = ['stop', 'up']

test_ds = MSCDataset('data/msc-test', CLASSES, torch.nn.Identity())

## Define the Model Name

In [12]:
MODEL_NAME = '1762361604'  # Enter the name of the model under evaluation

## Evaluate the ONNX Model

In [13]:
frontend_float32_file = f'saved_models/{MODEL_NAME}_frontend.onnx'
model_float32_file = f'saved_models/{MODEL_NAME}_model.onnx'
ort_frontend = ort.InferenceSession(frontend_float32_file)
ort_model = ort.InferenceSession(model_float32_file)

true_count = 0.0
for sample in test_ds:
    inputs = sample['x']
    label = sample['label']
    inputs = inputs.numpy()
    inputs = np.expand_dims(inputs, 0)
    features = ort_frontend.run(None, {'input': inputs})[0]
    outputs = ort_model.run(None,  {'input': features})[0]
    prediction = np.argmax(outputs, axis=-1).item()
    true_count += prediction == label

float32_accuracy = true_count / len(test_ds) * 100
frontend_size = os.path.getsize(frontend_float32_file)
model_float32_size = os.path.getsize(model_float32_file)
total_float32_size = frontend_size + model_float32_size

print(f'Float32 Accuracy: {float32_accuracy:.2f}%')
print(f'Float32 Frontend Size: {frontend_size / 2**10:.1f}KB')
print(f'Float32 Model Size: {model_float32_size / 2**10:.1f}KB')
print(f'Float32 Total Size: {total_float32_size / 2**10:.1f}KB')

RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, and 7 on all platforms, and 8 on Mac and Linux.
          2. The PyTorch version (2.9.1+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.
        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8: Could not load this library: C:\Users\andre\Documents\University\2nd Year\EC4AI\efficient-computing-for-artificial-intelligence\.venv\Lib\site-packages\torchcodec\libtorchcodec_core8.dll
FFmpeg version 7: Could not load this library: C:\Users\andre\Documents\University\2nd Year\EC4AI\efficient-computing-for-artif

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=3880e510-b64c-4bb5-b488-c2122d5d9e2d' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>